# ECABSD V3: 1000+ Complex Training Pipeline
This notebook trains the **Equivariant Cross-Attention Binding Site Detector (ECABSD)** on a large-scale dataset (DIPS) while strictly preventing data leakage into the DB5 benchmark test set using MMSeqs2 (30% identity threshold).

### Instructions:
1. Ensure you have attached your raw DIPS PDB dataset to this Kaggle notebook (e.g., at `/kaggle/input/dips-dataset`).
2. Ensure **GPU** and **Internet** are enabled in the notebook settings.
3. Run the cells below sequentially.

### 1. Install Dependencies
Install deep learning and bioinformatics libraries.

In [ ]:
!pip install -q transformers sentence-transformers
!pip install -q torch-geometric
!conda install -c conda-forge -c bioconda mmseqs2 -y

### 2. Prepare DB5 Benchmark (Test Set)
Download and extract physical 5.0Å graphs for the DB5 benchmark.

In [ ]:
!python scripts/download_benchmarks.py --output-dir /kaggle/working/db5_raw
!python scripts/prepare_dataset.py \
    --pdb-dir /kaggle/working/db5_raw \
    --output-dir /kaggle/working/db5_graphs \
    --cutoff 5.0 \
    --threads 4

### 3. Process Massive DIPS Dataset
Extract 5.0Å graphs from the attached DIPS dataset, applying rigorous filters (<30 res, >512 res, missing coords).

*Note: Update the `--pdb-dir` below if your attached Kaggle dataset has a different name!*

In [ ]:
DIPS_INPUT_DIR = "/kaggle/input/dips-dataset"

!python scripts/prepare_kaggle_dips.py \
    --pdb-dir {DIPS_INPUT_DIR} \
    --output-dir /kaggle/working/dips_graphs \
    --cutoff 5.0 \
    --threads 4

### 4. Strict Leakage Removal (MMSeqs2)
Quarantine the DIPS dataset against DB5. Any DIPS complex sharing >30% sequence identity with DB5 is discarded.

In [ ]:
!python scripts/filter_dips_mmseqs.py \
    --dips-splits /kaggle/working/dips_graphs/splits.csv \
    --dips-pdb-dir {DIPS_INPUT_DIR} \
    --db5-splits /kaggle/working/db5_graphs/splits.csv \
    --db5-pdb-dir /kaggle/working/db5_raw \
    --output-csv /kaggle/working/dips_graphs/splits_cleaned.csv \
    --threshold 0.30

### 5. Train ECABSD V3
Launch the GPU training using the mathematically clean training splits.

In [ ]:
!python train_v3.py \
    --graph-dir /kaggle/working/dips_graphs \
    --splits /kaggle/working/dips_graphs/splits_cleaned.csv \
    --epochs 50 \
    --batch-size 32 \
    --learning-rate 1e-4